# Module 6 — Object-Oriented Programming and Advanced Python

**Python for Machine Learning · MSc Artificial Intelligence · Swansea University**

---

## Learning objectives

1. Explain classes, objects, attributes and methods.
2. Write classes with `__init__`, instance and class attributes, and special ("dunder") methods.
3. Use **inheritance**, `super()` and abstract base classes — the design behind scikit-learn and PyTorch.
4. Use `@property`, `@staticmethod`, `@classmethod` and `@dataclass`.
5. Understand **iterators, generators, decorators and context managers**, which appear throughout ML libraries.

---

## 6.1 Why object-oriented programming?

Everything in Python is an **object**: numbers, strings, lists, functions, modules. An object bundles

* **data** (its *attributes*), and
* **behaviour** (its *methods* — functions attached to it).

In [1]:
nums = [3, 1, 2]
nums.sort()                     # a method of the list object
print(nums, type(nums))
print((3.5).is_integer())       # even a float literal has methods

[1, 2, 3] <class 'list'>
False


A **class** is a blueprint; an **object** (or *instance*) is a concrete thing built from it. Every ML library is built this way:

```text
model = RandomForestClassifier(n_estimators=100)   # create an object (instance)
model.fit(X_train, y_train)                        # method that changes its state
model.predict(X_test)                              # method that uses that state
model.feature_importances_                         # attribute learned during fit
```

Understanding classes lets you read library code, debug it, and write your own models, datasets and pipelines.

## 6.2 Defining a class

In [2]:
class Dog:
    species = "Canis familiaris"          # class attribute: shared by all dogs

    def __init__(self, name, age):        # the initialiser, runs when an object is created
        self.name = name                  # instance attributes: one copy per object
        self.age = age

    def bark(self):                       # a method
        return f"{self.name} says woof!"

    def birthday(self):
        self.age += 1                     # methods can modify the object's state

rex = Dog("Rex", 3)
fido = Dog("Fido", 5)
print(rex.bark())
rex.birthday()
print(rex.name, rex.age, "|", fido.name, fido.age)
print(rex.species, fido.species)

Rex says woof!
Rex 4 | Fido 5
Canis familiaris Canis familiaris


**What is `self`?** It is the object the method is called on. `rex.bark()` is translated by Python into `Dog.bark(rex)`, so inside the method `self` is `rex`. You must write `self` as the first parameter of every (instance) method.

In [3]:
print(Dog.bark(fido))      # equivalent to fido.bark()
print(isinstance(rex, Dog))

Fido says woof!
True


### Class attributes vs instance attributes

Class attributes are shared. Be careful with *mutable* class attributes — the same trap as mutable default arguments:

In [4]:
class Bad:
    history = []                  # ONE list shared by every instance
    def log(self, x):
        self.history.append(x)

class Good:
    def __init__(self):
        self.history = []         # each instance gets its own list
    def log(self, x):
        self.history.append(x)

a, b = Bad(), Bad()
a.log(1)
print("Bad:", b.history)
c, d = Good(), Good()
c.log(1)
print("Good:", d.history)

Bad: [1]
Good: []


## 6.3 A first ML class: a simple linear regressor

Let us write a model with the same interface as scikit-learn: hyper-parameters in `__init__`, learning in `fit`, inference in `predict`. It fits $y = wx + b$ by ordinary least squares:

$$w = \frac{\sum (x_i-\bar x)(y_i - \bar y)}{\sum (x_i - \bar x)^2}, \qquad b = \bar y - w\bar x$$

In [5]:
class SimpleLinearRegression:
    """One-feature linear regression fitted by least squares."""

    def __init__(self, fit_intercept=True):
        self.fit_intercept = fit_intercept   # hyper-parameter (set by the user)
        self.coef_ = None                    # learned parameters: trailing underscore,
        self.intercept_ = None               # as in scikit-learn

    def fit(self, x, y):
        n = len(x)
        x_mean = sum(x) / n if self.fit_intercept else 0.0
        y_mean = sum(y) / n if self.fit_intercept else 0.0
        num = sum((xi - x_mean) * (yi - y_mean) for xi, yi in zip(x, y))
        den = sum((xi - x_mean) ** 2 for xi in x)
        self.coef_ = num / den
        self.intercept_ = y_mean - self.coef_ * x_mean
        return self                          # returning self allows chaining: model.fit(...).predict(...)

    def predict(self, x):
        if self.coef_ is None:
            raise RuntimeError("Model is not fitted yet. Call fit() first.")
        return [self.coef_ * xi + self.intercept_ for xi in x]

    def score(self, x, y):
        """Coefficient of determination R^2."""
        y_hat = self.predict(x)
        y_mean = sum(y) / len(y)
        ss_res = sum((yi - pi) ** 2 for yi, pi in zip(y, y_hat))
        ss_tot = sum((yi - y_mean) ** 2 for yi in y)
        return 1 - ss_res / ss_tot

x = [1, 2, 3, 4, 5]
y = [2.1, 3.9, 6.2, 8.1, 9.8]
model = SimpleLinearRegression().fit(x, y)
print(f"w={model.coef_:.3f}, b={model.intercept_:.3f}")
print(model.predict([6, 7]))
print(f"R^2 = {model.score(x, y):.4f}")

w=1.960, b=0.140
[11.900000000000002, 13.86]
R^2 = 0.9976


## 6.4 Special ("dunder") methods

Methods with double underscores, such as `__init__`, let your objects work with Python's built-in syntax: `print()`, `len()`, `+`, `==`, `[]`, `for` loops…

| Method | Triggered by |
|---|---|
| `__repr__` / `__str__` | `repr(obj)`, the notebook display / `print(obj)`, `str(obj)` |
| `__len__` | `len(obj)` |
| `__getitem__` | `obj[i]` |
| `__iter__` | `for x in obj` |
| `__eq__`, `__lt__` | `==`, `<` |
| `__add__`, `__mul__` | `+`, `*` |
| `__call__` | `obj(...)` — calling an object like a function |

In [6]:
import math

class Vector:
    def __init__(self, *components):
        self.components = list(components)

    def __repr__(self):                      # unambiguous representation for developers
        return f"Vector{tuple(self.components)}"

    def __len__(self):
        return len(self.components)

    def __getitem__(self, i):
        return self.components[i]

    def __add__(self, other):
        return Vector(*[a + b for a, b in zip(self.components, other.components)])

    def __mul__(self, scalar):               # vector * number
        return Vector(*[a * scalar for a in self.components])

    def __eq__(self, other):
        return self.components == other.components

    def dot(self, other):
        return sum(a * b for a, b in zip(self.components, other.components))

    def norm(self):
        return math.sqrt(self.dot(self))

u = Vector(1, 2, 3)
v = Vector(4, 5, 6)
print(u + v, u * 2, len(u), u[0])
print(u.dot(v), round(v.norm(), 3), u == Vector(1, 2, 3))
for comp in u:             # works because __getitem__ is defined
    print(comp, end=" ")

Vector(5, 7, 9) Vector(2, 4, 6) 3 1
32 8.775 True
1 2 3 

NumPy arrays are exactly this idea, implemented in C for speed (Module 7).

A **dataset** class with `__len__` and `__getitem__` is precisely the interface PyTorch's `Dataset` requires:

In [7]:
class ToyDataset:
    def __init__(self, features, labels):
        assert len(features) == len(labels)
        self.features, self.labels = features, labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

ds = ToyDataset([[0.1, 0.2], [0.3, 0.4], [0.5, 0.6]], [0, 1, 0])
print(len(ds), ds[1])

3 ([0.3, 0.4], 1)


`__call__` makes objects callable — in PyTorch, `model(x)` calls the model's `forward` method this way:

In [8]:
class Scaler:
    def __init__(self, factor):
        self.factor = factor
    def __call__(self, x):
        return x * self.factor

double = Scaler(2)
print(double(21))

42


## 6.5 Inheritance

A **subclass** inherits attributes and methods from a **parent** class, and can add or *override* them. `super()` calls the parent's version.

In [9]:
class Model:
    """Base class for all our models."""

    def __init__(self, name):
        self.name = name
        self.is_fitted = False

    def fit(self, X, y):
        raise NotImplementedError("subclasses must implement fit")

    def __repr__(self):
        status = "fitted" if self.is_fitted else "not fitted"
        return f"{self.__class__.__name__}(name={self.name!r}, {status})"


class MajorityClassifier(Model):
    """Always predicts the most frequent class seen during training (a baseline)."""

    def __init__(self):
        super().__init__(name="majority")    # run the parent's __init__
        self.majority_ = None

    def fit(self, X, y):                     # override the parent's method
        self.majority_ = max(set(y), key=y.count)
        self.is_fitted = True
        return self

    def predict(self, X):
        return [self.majority_ for _ in X]


clf = MajorityClassifier()
print(clf)
clf.fit([[1], [2], [3], [4]], ["cat", "dog", "cat", "cat"])
print(clf, clf.predict([[5], [6]]))
print(isinstance(clf, Model), issubclass(MajorityClassifier, Model))

MajorityClassifier(name='majority', not fitted)
MajorityClassifier(name='majority', fitted) ['cat', 'cat']
True True


A baseline like this is important in practice: if your sophisticated model can't beat "always predict the majority class", something is wrong.

### Abstract base classes

`abc.ABC` and `@abstractmethod` *force* subclasses to implement certain methods — you cannot even create an object of a class with unimplemented abstract methods:

In [10]:
from abc import ABC, abstractmethod

class Activation(ABC):
    @abstractmethod
    def __call__(self, x): ...

    @abstractmethod
    def derivative(self, x): ...

class ReLU(Activation):
    def __call__(self, x):
        return max(0.0, x)
    def derivative(self, x):
        return 1.0 if x > 0 else 0.0

class Sigmoid(Activation):
    def __call__(self, x):
        return 1 / (1 + math.exp(-x))
    def derivative(self, x):
        s = self(x)
        return s * (1 - s)

for act in [ReLU(), Sigmoid()]:          # polymorphism: same interface, different behaviour
    print(type(act).__name__, act(0.5), round(act.derivative(0.5), 4))

try:
    Activation()
except TypeError as e:
    print("TypeError:", e)

ReLU 0.5 1.0
Sigmoid 0.6224593312018546 0.235
TypeError: Can't instantiate abstract class Activation without an implementation for abstract methods '__call__', 'derivative'


**Polymorphism** — different classes sharing one interface — is why you can swap `LogisticRegression()` for `RandomForestClassifier()` in scikit-learn without changing any other line of code.

## 6.6 Encapsulation and properties

Python has no truly private attributes. By convention:

* `_name` (one underscore) means "internal — please don't touch from outside";
* `__name` (two underscores) triggers *name mangling*, making accidental access harder.

`@property` lets you expose a computed or validated attribute with normal attribute syntax:

In [11]:
class Optimizer:
    def __init__(self, lr):
        self.lr = lr                 # goes through the setter below

    @property
    def lr(self):
        return self._lr

    @lr.setter
    def lr(self, value):
        if value <= 0:
            raise ValueError("learning rate must be positive")
        self._lr = value

opt = Optimizer(0.01)
print(opt.lr)
opt.lr = 0.001
print(opt.lr)
try:
    opt.lr = -1
except ValueError as e:
    print("ValueError:", e)

0.01
0.001
ValueError: learning rate must be positive


## 6.7 `@staticmethod` and `@classmethod`

* A **static method** doesn't use `self` or the class — just a function grouped inside the class.
* A **class method** receives the class (`cls`) instead of an instance. Common use: *alternative constructors*.

In [12]:
class Experiment:
    def __init__(self, name, lr, epochs):
        self.name, self.lr, self.epochs = name, lr, epochs

    @classmethod
    def from_dict(cls, d):
        return cls(d["name"], d["lr"], d["epochs"])

    @staticmethod
    def is_valid_lr(lr):
        return 0 < lr < 1

    def __repr__(self):
        return f"Experiment({self.name!r}, lr={self.lr}, epochs={self.epochs})"

exp = Experiment.from_dict({"name": "baseline", "lr": 0.01, "epochs": 20})
print(exp, Experiment.is_valid_lr(0.01))

Experiment('baseline', lr=0.01, epochs=20) True


## 6.8 Dataclasses

For classes that mainly *store data*, `@dataclass` writes `__init__`, `__repr__` and `__eq__` for you:

In [13]:
from dataclasses import dataclass, field

@dataclass
class TrainingConfig:
    model: str
    lr: float = 0.001
    epochs: int = 10
    layers: list[int] = field(default_factory=lambda: [64, 32])   # safe mutable default

    def total_units(self):
        return sum(self.layers)

cfg = TrainingConfig("MLP", epochs=50)
print(cfg)
print(cfg.total_units(), cfg == TrainingConfig("MLP", epochs=50))

TrainingConfig(model='MLP', lr=0.001, epochs=50, layers=[64, 32])
96 True


`@dataclass(frozen=True)` makes instances immutable.

---

## 6.9 Iterators and generators

An **iterable** is anything you can loop over. Under the hood, `for` calls `iter()` to get an **iterator**, then `next()` repeatedly until a `StopIteration` exception:

In [14]:
it = iter([10, 20, 30])
print(next(it), next(it), next(it))
try:
    next(it)
except StopIteration:
    print("iterator exhausted")

10 20 30
iterator exhausted


A **generator function** uses `yield` instead of `return`. Each `yield` produces one value and *pauses* the function until the next value is requested. Values are produced **lazily**, so a generator can represent huge — or even infinite — sequences with almost no memory.

In [15]:
def countdown(n):
    while n > 0:
        yield n
        n -= 1

print(list(countdown(5)))

gen = countdown(3)
print(next(gen), next(gen))

[5, 4, 3, 2, 1]
3 2


The perfect ML use case is **mini-batching** — splitting a dataset into batches for training:

In [16]:
def batches(data, batch_size):
    """Yield successive batches of `data`."""
    for start in range(0, len(data), batch_size):
        yield data[start:start + batch_size]

for batch in batches(list(range(10)), batch_size=4):
    print(batch)

[0, 1, 2, 3]
[4, 5, 6, 7]
[8, 9]


Reading a huge file line by line with a generator keeps memory constant:

```text
def read_records(path):
    with open(path) as f:
        for line in f:
            yield line.strip().split(",")
```

## 6.10 Decorators

A **decorator** is a function that takes a function and returns a modified version. The syntax `@decorator` above a `def` is shorthand for `func = decorator(func)`. You have already used some: `@property`, `@classmethod`, `@dataclass`, `@lru_cache`. In PyTorch you'll see `@torch.no_grad()`.

In [17]:
import time
import functools

def timer(func):
    @functools.wraps(func)          # keeps the original name and docstring
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {time.perf_counter() - start:.4f} s")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(i * i for i in range(n))

print(slow_sum(1_000_000))

slow_sum took 0.0535 s
333332833333500000


A decorator that takes arguments needs one more level of nesting:

In [18]:
def retry(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except ConnectionError as e:
                    print(f"attempt {attempt} failed: {e}")
            raise RuntimeError(f"failed after {times} attempts")
        return wrapper
    return decorator

calls = {"n": 0}

@retry(times=3)
def flaky_download():
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("network glitch")
    return "data downloaded"

print(flaky_download())

attempt 1 failed: network glitch
attempt 2 failed: network glitch
data downloaded


## 6.11 Context managers

The `with` statement guarantees set-up and clean-up code runs, even if errors occur. You have used it with `open()`. You can write your own with `contextlib.contextmanager` — code before `yield` is the set-up, code after it is the clean-up:

In [19]:
from contextlib import contextmanager

@contextmanager
def timed_block(label):
    start = time.perf_counter()
    try:
        yield
    finally:
        print(f"[{label}] {time.perf_counter() - start:.4f} s")

with timed_block("training"):
    total = sum(i ** 0.5 for i in range(500_000))

[training] 0.0622 s


In ML you'll meet `with torch.no_grad():` (disable gradient tracking) and `with warnings.catch_warnings():`.

---

## Summary

* A class bundles data (attributes) and behaviour (methods); `self` is the current object.
* Dunder methods (`__repr__`, `__len__`, `__getitem__`, `__add__`, `__call__`…) integrate objects with Python syntax.
* Inheritance and abstract base classes give many classes one common interface (polymorphism) — the scikit-learn `fit/predict` design.
* `@property`, `@classmethod`, `@staticmethod`, `@dataclass` are common class tools.
* Generators (`yield`) produce values lazily; decorators wrap functions; context managers (`with`) guarantee clean-up.

---

## Exercises

Solutions: `solutions/06_oop_and_advanced_python_solutions.ipynb`.

**Exercise 6.1 — `BankAccount`.**
Write a class with attributes `owner` and `balance` (default 0), methods `deposit(amount)` and `withdraw(amount)`. Both should raise `ValueError` for non-positive amounts, and `withdraw` should raise `ValueError` if funds are insufficient. Keep a list of all transactions and add a `__repr__`.

In [20]:
# Your code here

**Exercise 6.2 — A `StandardScaler` class.**
Write a class `StandardScaler` with methods `fit(values)` (stores `mean_` and `std_`, returns `self`), `transform(values)` (returns $(x - \mu)/\sigma$), and `fit_transform(values)`. `transform` must raise a `RuntimeError` if called before `fit`. Fit on `train` and transform `test` — **why must we use the training mean and std for the test data?** Answer in a comment.

In [21]:
train = [10, 12, 14, 16, 18]
test = [11, 20]
# Your code here

**Exercise 6.3 — Inheritance: shapes.**
Create an abstract class `Shape` with abstract methods `area()` and `perimeter()`. Implement `Rectangle(w, h)`, `Square(side)` (inherit from `Rectangle`!) and `Circle(r)`. Put several shapes in a list, sort them by area and print each with its area and perimeter.

In [22]:
# Your code here

**Exercise 6.4 — A k-Nearest-Neighbours classifier.**
Implement `KNNClassifier(k=3)` with `fit(X, y)` (just stores the data) and `predict(X)`. For each new point, compute the Euclidean distance to all training points, take the `k` closest, and predict the most common label among them. Test with the data below.

In [23]:
X_train = [[1, 1], [1, 2], [2, 1], [6, 6], [6, 7], [7, 6]]
y_train = ["A", "A", "A", "B", "B", "B"]
X_new = [[1.5, 1.5], [6.5, 6.5], [4, 4]]
# Your code here

**Exercise 6.5 — Generators.**
(a) Write a generator `fibonacci_gen()` that yields Fibonacci numbers **forever**; print the first 12 using `itertools.islice`. (b) Write a generator `sliding_window(seq, size)` that yields consecutive windows, e.g. `[1,2,3,4]` with size 2 → `[1,2]`, `[2,3]`, `[3,4]`. Use it to compute a moving average of `prices` with window 3.

In [24]:
import itertools
prices = [10, 11, 13, 12, 15, 18, 17]
# Your code here

**Exercise 6.6 — A decorator.**
Write a decorator `count_calls` that counts how many times a function is called, storing the count in an attribute `wrapper.calls`. Apply it to a function and show it works.

In [25]:
# Your code here

**Exercise 6.7 — Dataclass and dunder methods (challenge).**
Create a `@dataclass` `Result` with fields `model: str`, `accuracy: float`, `time_s: float`. Make results sortable by accuracy (hint: `@dataclass(order=True)` with a `sort_index` field set in `__post_init__`, **or** implement `__lt__` yourself). Sort a list of results and print the best one.

In [26]:
# Your code here